In [ ]:
from collections import defaultdict
from glob import glob
from pathlib import Path

import geopandas
import matplotlib.pyplot as plt
import numpy
import pandas

from snail.intersection import GridDefinition
from snail.damages import PiecewiseLinearDamageCurve
from tqdm.notebook import tqdm

In [ ]:
grid = GridDefinition.from_raster(
    "../inputs/fluvial_raw_fld_depth/JM_FLRF_UD_Q20_RD_02-aligned.tif"
)

In [ ]:
def nonzero_exposure(exposure, hazard_prefix):
    cols = [col for col in exposure.columns if hazard_prefix in col]
    for col in cols:
        exposure.loc[exposure[col] < 0, col] = 0
    exposure[f"{hazard_prefix}_max"] = exposure[cols].max(axis=1)
    return exposure[exposure[f"{hazard_prefix}_max"] > 0].copy()


def cell_index(df, grid):
    col_idx, row_idx = df.i_0.values, df.j_0.values
    nrows, ncols = grid.height, grid.width
    idx = numpy.ravel_multi_index((row_idx, col_idx), (nrows, ncols))
    return idx


rp_exposure = nonzero_exposure(
    geopandas.read_parquet("../outputs/exposure/buildings_flrf.parquet"), "flrf"
)
rp_exposure["cell_index"] = cell_index(rp_exposure, grid)
rp_exposure.PARISH = (
    rp_exposure.PARISH.str.replace(" ", "").str.replace("ST.", "ST. ").str.title()
)
rp_exposure.set_index("cell_index", inplace=True)

In [ ]:
event_depths = pandas.read_parquet(
    "../outputs/ObsEventRP/R06_2016R1_Obs_0000/ObsEventRP__FLRF__R06_2016R1_Obs_00000057.parquet"
)
event_depths.set_index("cell_index", inplace=True)

In [ ]:
building_event_exposure = rp_exposure.join(event_depths, how="left")[
    ["osm_id", "i_0", "j_0", "depth", "geometry"]
]

In [ ]:
building_event_exposure.to_parquet("../outputs/test.parquet")

In [ ]:
!pq2gpkg ../outputs/test.parquet ../outputs/test.gpkg test

In [ ]:
event_fnames = sorted(glob("../outputs/ObsEventRP/**/*FLRF*.parquet"))
len(event_fnames)

In [ ]:
rp_exposure.columns

In [ ]:
rp_exposure.cost_unit.unique()

In [ ]:
event_exposure = rp_exposure[
    [
        "osm_id",
        "ED_ID",
        "ED",
        "PARISH",
        "CONST_NAME",
        "building_type",
        "min_damage_cost",
        "max_damage_cost",
        "mean_damage_cost",
        "cost_unit",
        "total_GDP",
        "geometry",
    ]
].copy()

# Calculate area
event_exposure["exposed_area_m2"] = event_exposure.area

# Calculate unit repair/rehabilitation cost in Jamaican dollars
event_exposure.min_damage_cost = (
    event_exposure.min_damage_cost * event_exposure.exposed_area_m2
)
event_exposure.max_damage_cost = (
    event_exposure.max_damage_cost * event_exposure.exposed_area_m2
)
event_exposure.mean_damage_cost = (
    event_exposure.mean_damage_cost * event_exposure.exposed_area_m2
)

# Update unit
event_exposure["cost_unit"] = "JD"

for event_fname in tqdm(event_fnames):
    event_key = Path(event_fname).stem
    event_depths = (
        pandas.read_parquet(event_fname)
        .set_index("cell_index")[["depth"]]
        .rename(columns={"depth": event_key})
    )
    event_exposure = event_exposure.join(event_depths, how="left")

In [ ]:
event_exposure.columns

In [ ]:
event_exposure.to_parquet("../outputs/exposure/buildings_flrf_event_depths.parquet")

In [ ]:
curves = {}
for sector in ("industrial", "commercial", "residential"):
    for sensitivity in ("damage_ratio", "damage_ratio_min", "damage_ratio_max"):
        curves[(sector, sensitivity)] = PiecewiseLinearDamageCurve.from_excel(
            "../inputs/damage_curves/damage_curves_buildings_flooding.xlsx",
            sector,
            intensity_col="flood_depth",
            damage_col=sensitivity,
        )

In [ ]:
building_sector = pandas.read_csv(
    "../inputs/damage_curves/asset_damage_curve_mapping.csv"
)[["asset_name", "asset_sheet"]]
curve_sector_lookup = defaultdict(list)
for b in building_sector.itertuples():
    curve_sector_lookup[b.asset_sheet].append(b.asset_name)

In [ ]:
dict(curve_sector_lookup)

In [ ]:
event_damage = event_exposure.copy()
event_cols = [col for col in event_damage.columns if "FLRF" in col]
for sector in ("industrial", "commercial", "residential"):
    row_mask = event_damage.building_type.isin(curve_sector_lookup[sector])
    depths = event_damage.loc[row_mask, event_cols]
    damage_ratios = curves[(sector, "damage_ratio")].damage_fraction(depths)
    damage_costs = (
        damage_ratios * event_damage.loc[row_mask, "mean_damage_cost"].values[:, None]
    )
    event_damage.loc[row_mask, event_cols] = damage_costs

In [ ]:
event_damage[["mean_damage_cost", "ObsEventRP__FLRF__R06_2016R1_Obs_00000057"]].dropna()

In [ ]:
len(event_damage.osm_id.unique()), len(event_damage.osm_id.unique()) / len(event_damage)

In [ ]:
event_damage

In [ ]:
building_event_damage = (
    event_damage.drop(
        columns=[
            "min_damage_cost",
            "max_damage_cost",
            "mean_damage_cost",
            "cost_unit",
            "total_GDP",
            "geometry",
            "exposed_area_m2",
        ]
    )
    .groupby(["osm_id", "ED_ID", "ED", "PARISH", "CONST_NAME", "building_type"])
    .sum()
    .reset_index()
)

In [ ]:
for sector in ("industrial", "commercial", "residential"):
    row_mask = building_event_damage.building_type.isin(curve_sector_lookup[sector])
    building_event_damage.loc[row_mask, "building_sector"] = sector

In [ ]:
building_event_damage

In [ ]:
parish_event_damage = (
    building_event_damage.drop(
        columns=[
            "osm_id",
            "ED_ID",
            "ED",
            "CONST_NAME",
            "building_type",
            "building_sector",
        ]
    )
    .groupby(["PARISH"])
    .sum()
)
parish_event_damage

In [ ]:
parishes = geopandas.read_file("../inputs/admin_boundaries.gpkg", layer="admin1")[
    ["PARISH", "geometry"]
].set_index("PARISH")
parishes

In [ ]:
parish_event_damage_geo = parishes.join(parish_event_damage)

In [ ]:
!mkdir -p ../outputs/figures

In [ ]:
track_ids = []
obs_events = pandas.read_csv("../inputs/event_data/ObsEventInfo.csv")
for track_str in obs_events["track.id"].dropna().values:
    tracks = track_str.split(", ")
    track_ids.extend(tracks)
obs_event_track_ids = obs_events[
    ["event.id", "track.id", "start.year", "start.month", "duration"]
].set_index("event.id")
obs_event_track_ids.head(2)

In [ ]:
track_info = pandas.read_csv(
    "../inputs/event_data/tracks_na.tsv",
    sep="\t",
    na_values=None,
    keep_default_na=False,
)
track_info = track_info[track_info["track.id"].isin(track_ids)].set_index("track.id")
track_info.to_csv("../inputs/event_data/track_info.csv")

In [ ]:
track_info

In [ ]:
for event in event_cols:
    event_id = event.replace("ObsEventRP__FLRF__", "")
    track_str = obs_event_track_ids.loc[event_id, "track.id"]
    try:
        len(track_str)
        track_ids = track_str.split(", ")
        track_names = [track_info.loc[track_id, "name"] for track_id in track_ids]
        title = f"{event_id}\n{', '.join(track_ids)} ({', '.join(track_names)})"
    except:
        title = event_id

    fig, ax = plt.subplots()
    ax.set_title(title)
    parish_event_damage_geo.plot(
        ax=ax,
        column=event,
        legend=True,
        legend_kwds={
            "label": "River flooding direct damage to buildings (J$)",
            "orientation": "horizontal",
        },
    )
    ax.set_axis_off()
    plt.savefig(f"../outputs/figures/{event}_parish_building_damages.png")
    plt.close()

In [ ]:
total_event_damage = building_event_damage.drop(
    columns=[
        "osm_id",
        "ED_ID",
        "ED",
        "CONST_NAME",
        "building_type",
        "building_sector",
        "PARISH",
    ]
).sum()
total_event_damage = pandas.DataFrame(total_event_damage).reset_index()
total_event_damage.columns = ["event.id", "total_building_damage_JD"]
total_event_damage["event.id"] = total_event_damage["event.id"].str.replace(
    "ObsEventRP__FLRF__", ""
)
# total_event_damage.set_index('event.id', inplace=True)
# total_event_damage = total_event_damage.join(obs_event_track_ids).reset_index().merge(track_info, on='track.id', how='left').fillna("")

# total_event_damage["event.title"] = total_event_damage['name'] \
#     + ' ' \
#     + total_event_damage['start.year'].astype("str") \
#     + '-'  \
#     + total_event_damage['start.month'].astype("str")
# total_event_damage
total_event_damage

In [ ]:
total_event_damage.set_index("event.title")[["total_building_damage_JD"]].plot.bar()